# TP BIM → Neo4j → Python

Il part d'un petit dataset de pièces de bâtiment, le charge dans Neo4j, puis applique **BFS**, **DFS**, **Dijkstra** et **la détection de cycles**

## 1. Prérequis
Installer les dépendances dans l'environnement Python du notebook :
```bash
pip install neo4j ifcopenshell networkx matplotlib
```
Lancer Neo4j localement puis adapter les paramètres de connexion ci-dessous.

## 2. Créer un script qui permet de se connecter à la base Neo4J

In [2]:
from neo4j import GraphDatabase
from collections import deque
import heapq

URI = 'bolt://neo4j:7687'
AUTH = ('neo4j', 'neo4j123')
driver = GraphDatabase.driver(URI, auth=AUTH)

## 3. Créer un script permettant de lire un ifc et puis créer des noeuds associés à IfcSpace.

In [3]:
import ifcopenshell

model = ifcopenshell.open("/home/jovyan/work/data/tp_bim_graph.ifc")

with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
    for space in model.by_type("IfcSpace"):
        session.run(
            """
            MERGE (s:Space {id:$id})
            SET s.name=$name
            """,
            id=space.GlobalId,
            name=space.Name
        )

## 4. Créer un script permettant de créer les connexions entre les noeuds de Space.

In [4]:
connections = [
    ("Hall", "Salle A"),
    ("Hall", "Salle B"),
    ("Hall", "Escalier"),
    ("Escalier", "Salle C")
]

with driver.session() as session:
    for a, b in connections:
        session.run(
            """
            MATCH (x:Space {name:$a})
            MATCH (y:Space {name:$b})
            MERGE (x)-[:CONNECTED_TO]->(y)
            MERGE (y)-[:CONNECTED_TO]->(x)
            """,
            a=a,
            b=b
        )

## 5. Créer un dataframe qui représente l'ifcproduit

In [5]:
import pandas as pd
import ifcopenshell

model = ifcopenshell.open("/home/jovyan/work/data/tp_bim_graph.ifc")

rows = []

for obj in model.by_type("IfcProduct"):
    rows.append({
        "Type": obj.is_a(),
        "GlobalId": obj.GlobalId,
        "Name": obj.Name
    })

df = pd.DataFrame(rows)

df

,Type,GlobalId,Name
0,IfcBuilding,3hTPBuilding000000000001,Batiment TP
1,IfcBuildingStorey,3hTPStorey0000000000001,RDC
2,IfcSite,3hTPSite000000000000001,Campus Demo
3,IfcSpace,3hTPSpaceHall00000000001,Hall
4,IfcSpace,3hTPSpaceSA000000000001,Salle A
5,IfcSpace,3hTPSpaceSB000000000001,Salle B
6,IfcSpace,3hTPSpaceEsc00000000001,Escalier
7,IfcSpace,3hTPSpaceSC000000000001,Salle C


## 6. Implémenter le script permettant de relire le graphe dans Python

In [7]:
import networkx as nx

G = nx.Graph()

with driver.session() as session:
    result = session.run(
        """
        MATCH (a:Space)-[:CONNECTED_TO]->(b:Space)
        RETURN a.name AS a, b.name AS b
        """
    )
    for record in result:
        G.add_edge(record["a"], record["b"])

print("Nœuds :", list(G.nodes))
print("Arêtes :", list(G.edges))

Nœuds : ['Escalier', 'Hall', 'Salle B', 'Salle A', 'Salle C']
Arêtes : [('Escalier', 'Hall'), ('Escalier', 'Salle C'), ('Hall', 'Salle B'), ('Hall', 'Salle A')]


## 7.1 Implémenter BFS pour determiner l'ordre de parcours depuis Hall(astuce : eviter la duplication des noeuds, c'est à dire éviter de rajouter les noeuds déja visités)

In [10]:
from collections import deque

def bfs(graph, start):
    visited = {start}         
    queue   = deque([start])   
    order   = []               

    while queue:
        node = queue.popleft()
        order.append(node)

        for neighbor in graph.neighbors(node):
            if neighbor not in visited:     
                visited.add(neighbor)
                queue.append(neighbor)

    return order

## 7.2. Quel est l'ordre de parcours BFS depuis `Hall` ?

In [24]:
bfs_order = bfs(G, "Hall")
print("Ordre BFS depuis Hall :", bfs_order)

Ordre BFS depuis Hall : ['Hall', 'Escalier', 'Salle B', 'Salle A', 'Salle C']


BFS explore **niveau par niveau** :
- Niveau 0 : `Hall`
- Niveau 1 : `Escalier`, `Salle B`, `Salle A` (voisins directs de Hall)
- Niveau 2 : `Salle C` (voisin d'Escalier, pas encore visité)

## 8.1 Implémenter DFS pour determiner l'ordre de parcours depuis Hall(astuce : eviter la duplication des noeuds, c'est à dire éviter de rajouter les noeuds déja visités)

In [25]:
def dfs(graph, node, visited=None, order=None):
    if visited is None:
        visited = set()
    if order is None:
        order = []

    visited.add(node)     
    order.append(node)

    for neighbor in graph.neighbors(node):
        if neighbor not in visited:   # évite de revisiter
            dfs(graph, neighbor, visited, order)

    return order

## 8.2 Quel est l'ordre de parcours DFS depuis `Hall` ?

In [26]:
dfs_order = dfs(G, "Hall")
print("Ordre DFS depuis Hall :", dfs_order)

Ordre DFS depuis Hall : ['Hall', 'Escalier', 'Salle C', 'Salle B', 'Salle A']


Hall → Escalier → Salle C (on va au bout)
Puis retour → Salle B → Salle A

## 9.1 Implémenter Dijkstra pour determiner la distance entre le hall et les pieces voisins

In [27]:
import heapq

def dijkstra(graph, start):
    distances = {node: float('inf') for node in graph.nodes}
    distances[start] = 0

    heap = [(0, start)]

    while heap:
        dist_u, u = heapq.heappop(heap)

        # Si on a déjà trouvé mieux, on ignore
        if dist_u > distances[u]:
            continue

        for v in graph.neighbors(u):
            poids = 1   # arête non pondérée = poids 1
            nouvelle_dist = distances[u] + poids

            if nouvelle_dist < distances[v]:
                distances[v] = nouvelle_dist
                heapq.heappush(heap, (nouvelle_dist, v))

    return distances

In [28]:
dist = dijkstra(G, "Hall")
print("Distances depuis Hall :")
for node, d in dist.items():
    print(f"  Hall → {node} : {d}")

Distances depuis Hall :
  Hall → Escalier : 1
  Hall → Hall : 0
  Hall → Salle B : 1
  Hall → Salle A : 1
  Hall → Salle C : 2


## 9.2 Quelle est la distance minimale entre `Hall` et `Salle C` 

La distance minimale entre 'Hall' et 'Salle C' est : 2
- Il n'existe pas de chemin direct Hall → Salle C ;
le seul chemin passe obligatoirement par l'Escalier.

## 10.1 Détection de cycles

In [29]:
def has_cycle(graph):
    visited = set()

    def dfs_cycle(node, parent):
        visited.add(node)
        for neighbor in graph.neighbors(node):
            if neighbor not in visited:
                if dfs_cycle(neighbor, node):
                    return True
            elif neighbor != parent:
                return True
        return False

    for node in graph.nodes:
        if node not in visited:
            if dfs_cycle(node, None):
                return True

    return False


In [30]:
cycle = has_cycle(G)
print("Le graphe contient un cycle :", cycle)

Le graphe contient un cycle : False


## 10.2 Le graphe contient-il un cycle ? Pourquoi ?

*Non**, le graphe ne contient pas de cycle.

Pour qu'il y ait un cycle, il faudrait qu'un nœud soit accessible
par au moins deux chemins distincts depuis la source.
Ici, chaque pièce n'a qu'un seul chemin depuis Hall :
- `Salle A`, `Salle B`, `Escalier` sont uniquement reliés à Hall.
- `Salle C` est uniquement reliée à Escalier.


In [31]:
# Vérification avec NetworkX
print("Est un arbre (sans cycle) :", nx.is_tree(G))
print("Est connexe               :", nx.is_connected(G))
print("Nombre de nœuds           :", G.number_of_nodes())
print("Nombre d'arêtes           :", G.number_of_edges())
# Un arbre à n nœuds a exactement n-1 arêtes
# 5 nœuds → 4 arêtes → confirme l'absence de cycle

Est un arbre (sans cycle) : True
Est connexe               : True
Nombre de nœuds           : 5
Nombre d'arêtes           : 4
